#### Clustring Columns :
- customer_id
- monthly_income
- monthly_expense
- savings_rate
- avg_transaction_amount
- transaction_frequency
- food_ratio
- shopping_ratio
- travel_ratio
- entertainment_ratio
- other_ratio

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Users\shali\OneDrive\Desktop\FinanceX\data\FinanceX_transaction.csv")

#### Monthly_income

In [3]:
df['transaction_date'] = pd.to_datetime(df['transaction_date'])

In [4]:
df['month'] = df['transaction_date'].dt.to_period('M')
income = (
    df[df['transaction_direction'] == 'Credit']
    .groupby(['customer_id', 'month'])['transaction_amount']
    .sum()
    .reset_index(name='monthly_income')
)

#### Monthly Expense

In [5]:
expense = (
    df[df['transaction_direction'] == 'Debit']
    .groupby(['customer_id', 'month'])['transaction_amount']
    .sum()
    .reset_index(name='monthly_expense')
)

In [6]:
clustering_df = income.merge(
    expense,
    on=['customer_id', 'month'],
    how='outer'
)

In [7]:
clustering_df[['monthly_income', 'monthly_expense']] = (
    clustering_df[['monthly_income', 'monthly_expense']]
    .fillna(0)
)

#### Saving Rate

In [8]:
clustering_df['savings_rate'] = np.where(
    clustering_df['monthly_income'] > 0,
    (
        clustering_df['monthly_income']
        - clustering_df['monthly_expense']
    ) / clustering_df['monthly_income'],
    0
)

#### Average transaction amount

In [9]:
avg_transaction = (
    df.groupby('customer_id')['transaction_amount']
    .mean()
    .reset_index(name='avg_transaction_amount')
)

In [10]:
clustering_df = clustering_df.merge(
    avg_transaction,
    on='customer_id',
    how='left'
)

#### Transaction frequency

In [11]:
valid_df = df[df['transaction_status'] == 'Success']

transaction_frequency = (
    valid_df.groupby('customer_id')
            .size()
            .reset_index(name='transaction_frequency')
)

In [12]:
clustering_df = clustering_df.merge(
    transaction_frequency,
    on='customer_id',
    how='left'
)

#### Food Ratio

In [13]:
# Valid expense transactions
expense_df = df[
    (df['transaction_status'] == 'Success') &
    (df['transaction_direction'] == 'Debit')
].copy()

# Total expense per customer
total_expense = (
    expense_df.groupby('customer_id')['transaction_amount']
    .sum()
    .reset_index(name='total_expense')
)

In [14]:
food_expense = (
    expense_df[expense_df['merchant_category'] == 'Food & Dining']
    .groupby('customer_id')['transaction_amount']
    .sum()
    .reset_index(name='food_expense')
)

In [15]:
clustering_df = clustering_df.merge(
    total_expense,
    on='customer_id',
    how='left'
)

clustering_df = clustering_df.merge(
    food_expense,
    on='customer_id',
    how='left'
)

In [16]:
# No food expense = 0
clustering_df['food_expense'] = clustering_df['food_expense'].fillna(0)

# Calculate food ratio
clustering_df['food_ratio'] = (
    clustering_df['food_expense'] /
    clustering_df['total_expense']
).fillna(0)

#### Shoping ratio

In [17]:
shopping_expense = (
    expense_df[expense_df['merchant_category'].isin([
        'Retail',
        'E-Commerce'
    ])]
    .groupby('customer_id')['transaction_amount']
    .sum()
    .reset_index(name='shopping_expense')
)

In [18]:
clustering_df = clustering_df.merge(
    shopping_expense,
    on='customer_id',
    how='left'
)

In [19]:
clustering_df['shopping_expense'] = (
    clustering_df['shopping_expense'].fillna(0)
)

In [20]:
clustering_df['shopping_ratio'] = (
    clustering_df['shopping_expense'] /
    clustering_df['total_expense']
).fillna(0)

#### travel_ratio

In [21]:
travel_expense = (
    expense_df[expense_df['merchant_category'] == 'Travel']
    .groupby('customer_id')['transaction_amount']
    .sum()
    .reset_index(name='travel_expense')
)

In [22]:
clustering_df = clustering_df.merge(
    travel_expense,
    on='customer_id',
    how='left'
)

In [23]:
clustering_df['travel_expense'] = (
    clustering_df['travel_expense'].fillna(0)
)

In [24]:
clustering_df['travel_ratio'] = (
    clustering_df['travel_expense'] /
    clustering_df['total_expense']
).fillna(0)

#### Entertenment ratio

In [25]:
entertainment_expense = (
    expense_df[expense_df['merchant_category'] == 'Entertainment']
    .groupby('customer_id')['transaction_amount']
    .sum()
    .reset_index(name='entertainment_expense')
)

In [26]:
clustering_df = clustering_df.merge(
    entertainment_expense,
    on='customer_id',
    how='left'
)

In [27]:
clustering_df['entertainment_expense'] = (
    clustering_df['entertainment_expense'].fillna(0)
)

In [28]:
clustering_df['entertainment_ratio'] = (
    clustering_df['entertainment_expense'] /
    clustering_df['total_expense']
).fillna(0)

In [29]:
clustering_df[
    [
        'customer_id',
        'entertainment_expense',
        'total_expense',
        'entertainment_ratio'
    ]
].head()

,customer_id,entertainment_expense,total_expense,entertainment_ratio
0,CUST000001,0.0,2947.67,0.0
1,CUST000001,0.0,2947.67,0.0
2,CUST000001,0.0,2947.67,0.0
3,CUST000001,0.0,2947.67,0.0
4,CUST000001,0.0,2947.67,0.0


#### Other_ratio

In [31]:
category_map = {
    'Food & Dining': 'Food',
    'Retail': 'Shopping',
    'E-Commerce': 'Shopping',
    'Travel': 'Travel',
    'Entertainment': 'Entertainment',
    'Utilities': 'Others',
    'Healthcare': 'Others',
    'Education': 'Others',
    'Fuel': 'Others',
    'Insurance': 'Others',
    'Government': 'Others',
    'Real Estate': 'Others'
}

expense_df['category_group'] = (
    expense_df['merchant_category']
    .map(category_map)
    .fillna('Others')
)

In [32]:
others_expense = (
    expense_df[expense_df['category_group'] == 'Others']
    .groupby('customer_id')['transaction_amount']
    .sum()
    .reset_index(name='others_expense')
)

In [33]:
clustering_df = clustering_df.merge(
    others_expense,
    on='customer_id',
    how='left'
)

In [34]:
clustering_df['others_expense'] = (
    clustering_df['others_expense'].fillna(0)
)

In [35]:
clustering_df['others_ratio'] = (
    clustering_df['others_expense'] /
    clustering_df['total_expense']
).fillna(0)

#### Feature selection

In [39]:
clustering_features = [
    'monthly_income',
    'monthly_expense',
    'savings_rate',
    'avg_transaction_amount',
    'transaction_frequency',
    'food_ratio',
    'shopping_ratio',
    'travel_ratio',
    'entertainment_ratio',
    'others_ratio'
]

In [40]:
cluster_df = clustering_df[clustering_features].copy()

In [41]:
cluster_df.head()

,monthly_income,monthly_expense,savings_rate,avg_transaction_amount,transaction_frequency,food_ratio,shopping_ratio,travel_ratio,entertainment_ratio,others_ratio
0,0.00,892.86,0.0,655.316,5.0,0.0,0.339251,0.0,0.0,0.660749
1,328.91,0.00,1.0,655.316,5.0,0.0,0.339251,0.0,0.0,0.660749
2,0.00,1000.00,0.0,655.316,5.0,0.0,0.339251,0.0,0.0,0.660749
3,0.00,584.18,0.0,655.316,5.0,0.0,0.339251,0.0,0.0,0.660749
4,0.00,470.63,0.0,655.316,5.0,0.0,0.339251,0.0,0.0,0.660749


In [47]:
cluster_df.isnull().sum()

monthly_income            0
monthly_expense           0
savings_rate              0
avg_transaction_amount    0
transaction_frequency     0
food_ratio                0
shopping_ratio            0
travel_ratio              0
entertainment_ratio       0
others_ratio              0
dtype: int64

In [46]:
cluster_df['transaction_frequency'] = (
    cluster_df['transaction_frequency'].fillna(0)
)

___

In [49]:
df.to_csv(r"C:\Users\shali\OneDrive\Desktop\FinanceX\data\clustering_dataset.csv", index=False)